In [4]:
# 쇼핑몰 고객 리뷰 데이터 분석 - CounterVectorizer
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer    # 단어 등장 횟수 기반 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer    # 단어 중요도 기반 벡터화
from sklearn.metrics.pairwise import cosine_similarity         # 문서 간 유사도 계산

reviews = [
    "배송이 빠르고 포장이 깔끔해서 만족했습니다",
    "상품 품질은 좋았지만 가격이 조금 비쌌습니다",
    "주문한 색상과 다른 제품이 도착했습니다",
    "고객센터 응답이 늦어서 불편했습니다",
    "교환 처리가 빠르게 진행되어 만족했습니다",
    "앱에서 결제가 자꾸 실패해서 주문을 못 했습니다",
    "재구매하고 싶을 만큼 제품이 마음에 들었습니다",
    "배송이 지연되어 선물 날짜를 맞추지 못했습니다",
    "환불 처리가 너무 오래 걸려서 불만족스러웠습니다",
    "제품 설명과 실제 상품이 달라서 실망했습니다",
    "쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다",
    "포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다",
    "고객센터 상담원이 친절하게 문제를 해결해 주었습니다",
    "반품 신청 과정이 복잡해서 이용하기 어려웠습니다",
]

review_df = pd.DataFrame({
    'review_id': range(1, len(reviews) + 1),
    'review_text': reviews
})

print(review_df)

    review_id                     review_text
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다
2           3           주문한 색상과 다른 제품이 도착했습니다
3           4             고객센터 응답이 늦어서 불편했습니다
4           5          교환 처리가 빠르게 진행되어 만족했습니다
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다
9          10        제품 설명과 실제 상품이 달라서 실망했습니다
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다
13         14      반품 신청 과정이 복잡해서 이용하기 어려웠습니다


In [5]:
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(review_df['review_text'])
# fit_transform: 어휘집 학습 + 벡터 변환 동시 수행
# print(count_matrix)   # (0, 26)  1 ...  ← 희소 행렬(sparse matrix) 형태

count_words = count_vectorizer.get_feature_names_out()
# 어휘집에서 추출된 단어 목록 반환
# ['가격이' '걸려서' '결제' '결제가' '고객센터' ...]

count_df = pd.DataFrame(count_matrix.toarray(), columns=count_words)
print(count_df.head(3))

# 원본 리뷰와 단어 등장 횟수 DataFrame 합치기
count_result = pd.concat([review_df[["review_id", "review_text"]], count_df], axis=1)
print(count_result)

   가격이  걸려서  결제  결제가  고객센터  과정이  교환  금액이  깔끔해서  나왔습니다  ...  처리가  친절하게  쿠폰  \
0    0    0   0    0     0    0   0    0     1      0  ...    0     0   0   
1    1    0   0    0     0    0   0    0     0      0  ...    0     0   0   
2    0    0   0    0     0    0   0    0     0      0  ...    0     0   0   

   파손된  포장이  품질은  해결해  했습니다  환불  훼손되어  
0    0    1    0    0     0   0     0  
1    0    0    1    0     0   0     0  
2    0    0    0    0     0   0     0  

[3 rows x 74 columns]
    review_id                     review_text  가격이  걸려서  결제  결제가  고객센터  과정이  \
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다    0    0   0    0     0    0   
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다    1    0   0    0     0    0   
2           3           주문한 색상과 다른 제품이 도착했습니다    0    0   0    0     0    0   
3           4             고객센터 응답이 늦어서 불편했습니다    0    0   0    0     1    0   
4           5          교환 처리가 빠르게 진행되어 만족했습니다    0    0   0    0     0    0   
5           6      앱에서 결제가 자꾸 실패해서

In [6]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(review_df['review_text'])
# print(tfidf_matrix)   # (0, 26)  0.4199670161265011 ...  ← 0~1 실수 가중치

# tfidf_vectorizer에서 가져와야 함 (count_vectorizer 아님)
tfidf_words = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_words)
print(tfidf_df.head(3))

tfidf_result_df = pd.concat([review_df[["review_id", "review_text"]], tfidf_df], axis=1)
print(tfidf_result_df)

        가격이  걸려서   결제  결제가  고객센터  과정이   교환  금액이      깔끔해서  나왔습니다  ...  처리가  \
0  0.000000  0.0  0.0  0.0   0.0  0.0  0.0  0.0  0.485223    0.0  ...  0.0   
1  0.417061  0.0  0.0  0.0   0.0  0.0  0.0  0.0  0.000000    0.0  ...  0.0   
2  0.000000  0.0  0.0  0.0   0.0  0.0  0.0  0.0  0.000000    0.0  ...  0.0   

   친절하게   쿠폰  파손된       포장이       품질은  해결해  했습니다   환불  훼손되어  
0   0.0  0.0  0.0  0.419967  0.000000  0.0   0.0  0.0   0.0  
1   0.0  0.0  0.0  0.000000  0.417061  0.0   0.0  0.0   0.0  
2   0.0  0.0  0.0  0.000000  0.000000  0.0   0.0  0.0   0.0  

[3 rows x 74 columns]
    review_id                     review_text       가격이       걸려서        결제  \
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다  0.000000  0.000000  0.000000   
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다  0.417061  0.000000  0.000000   
2           3           주문한 색상과 다른 제품이 도착했습니다  0.000000  0.000000  0.000000   
3           4             고객센터 응답이 늦어서 불편했습니다  0.000000  0.000000  0.000000   
4           5        

In [7]:
# '배송이' 단어 기준으로 두 방식의 값 차이 비교
target_word = '배송이'

if target_word in count_df.columns:
    compare_df = pd.DataFrame({
        "review_id": review_df['review_id'],
        "review_text": review_df['review_text'],
        "Count_Vectorizer": count_df[target_word],           # 정수: 등장 횟수
        "Tfidf_Vectorizer": tfidf_df[target_word].round(3)  # 실수: TF-IDF 가중치
    })
    print(f'{target_word}의 단어 비교 --')
    print(compare_df)
else:
    print(f'{target_word} 단어가 단어 목록에 없어요')

배송이의 단어 비교 --
    review_id                     review_text  Count_Vectorizer  \
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다                 1   
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다                 0   
2           3           주문한 색상과 다른 제품이 도착했습니다                 0   
3           4             고객센터 응답이 늦어서 불편했습니다                 0   
4           5          교환 처리가 빠르게 진행되어 만족했습니다                 0   
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다                 0   
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다                 0   
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다                 1   
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다                 0   
9          10        제품 설명과 실제 상품이 달라서 실망했습니다                 0   
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다                 0   
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다                 0   
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다                 0   
13         14      반품 신청 과정이 복잡해서 이용하기 어려웠습니다   

In [8]:
new_review = "배송이 늦고 포장이 파손되어 불편했습니다"

# 기존 리뷰 + 새 리뷰를 합쳐서 같이 벡터화
# → 같은 어휘 사전(vocabulary)을 공유해야 벡터 비교가 가능
all_reviews = review_df["review_text"].tolist() + [new_review]

similarity_vectorizer = TfidfVectorizer()
similarity_matrix = similarity_vectorizer.fit_transform(all_reviews)
# print(similarity_matrix)   # (0, 27)  0.38965492653614037 ...

new_review_vector = similarity_matrix[-1]    # 새 리뷰 벡터 (마지막 행)

# 새 리뷰 vs 기존 리뷰 전체 코사인 유사도 계산
similarities = cosine_similarity(
    new_review_vector, similarity_matrix[:-1]  # [:-1]: 기존 리뷰들만 (새 리뷰 제외)
)
print(similarities)

similarity_df = pd.DataFrame({
    "review_id": review_df["review_id"],
    "기존리뷰": review_df["review_text"],
    "새리뷰와의 유사도": similarities[0]
})
# 유사도 높은 순으로 정렬
similarity_df = similarity_df.sort_values(by="새리뷰와의 유사도", ascending=False)
print(similarity_df)

[[0.30366192 0.         0.         0.20242753 0.         0.
  0.         0.12759078 0.         0.         0.         0.12217023
  0.         0.        ]]
    review_id                            기존리뷰  새리뷰와의 유사도
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다   0.303662
3           4             고객센터 응답이 늦어서 불편했습니다   0.202428
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다   0.127591
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다   0.122170
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다   0.000000
2           3           주문한 색상과 다른 제품이 도착했습니다   0.000000
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다   0.000000
4           5          교환 처리가 빠르게 진행되어 만족했습니다   0.000000
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다   0.000000
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다   0.000000
9          10        제품 설명과 실제 상품이 달라서 실망했습니다   0.000000
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다   0.000000
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다   0.000000
13         14      반품 신청 과정이 복잡해서 이용하기 어려웠습니다   

In [9]:
# 각 카테고리를 대표하는 키워드 문장
category_texts = [
    "배송 지연 늦음 도착 포장 파손 훼손",
    "결제 실패 쿠폰 적용 금액 오류 주문 앱",
    "교환 환불 반품 처리 오래 복잡 불편",
    "상품 품질 제품 설명 색상 다름 실망",   # ★ 쉼표 필수 (없으면 다음 문자열과 자동 결합됨)
    "만족 재구매 친절 해결 마음 좋음"
]

category_names = ["배송/포장 문제", "결제/주문 문제", "교환/환불/반품 문제", "상품 품질/정보 문제", "만족도 문제"]

# 리뷰 + 카테고리 대표 문장을 함께 벡터화 (같은 어휘 공간에서 비교)
texts_for_category = review_df['review_text'].tolist() + category_texts

category_vectorizer = TfidfVectorizer()
category_matrix = category_vectorizer.fit_transform(texts_for_category)

# 리뷰 벡터 / 카테고리 벡터 분리
review_vector = category_matrix[:len(review_df)]      # 앞쪽: 실제 리뷰
category_vectors = category_matrix[len(review_df):]   # 뒤쪽: 카테고리 대표 문장

# 각 리뷰 × 각 카테고리 유사도 행렬 (shape: 14 × 5)
category_similarity = cosine_similarity(review_vector, category_vectors)

classification_results = []

for i, review in enumerate(review_df['review_text']):
    best_category_index = category_similarity[i].argmax()   # 유사도 최대인 카테고리 인덱스
    best_score = category_similarity[i][best_category_index]
    classification_results.append({
        'review_id': review_df.loc[i, 'review_id'],
        'review_text': review,
        '분류결과': category_names[best_category_index],
        '유사도': round(best_score, 3)
    })

classification_df = pd.DataFrame(classification_results)
print('리뷰 분류 결과')
print(classification_df)

print('카테고리별 리뷰 갯수 집계')
category_count_df = classification_df['분류결과'].value_counts().reset_index()
category_count_df.columns = ['카테고리', '리뷰갯수']
print(category_count_df)

리뷰 분류 결과
    review_id                     review_text         분류결과    유사도
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다     배송/포장 문제  0.000
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다  상품 품질/정보 문제  0.104
2           3           주문한 색상과 다른 제품이 도착했습니다     배송/포장 문제  0.000
3           4             고객센터 응답이 늦어서 불편했습니다     배송/포장 문제  0.000
4           5          교환 처리가 빠르게 진행되어 만족했습니다  교환/환불/반품 문제  0.150
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다     배송/포장 문제  0.000
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다     배송/포장 문제  0.000
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다     배송/포장 문제  0.000
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다  교환/환불/반품 문제  0.271
9          10        제품 설명과 실제 상품이 달라서 실망했습니다  상품 품질/정보 문제  0.127
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다     결제/주문 문제  0.219
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다  상품 품질/정보 문제  0.099
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다     배송/포장 문제  0.000
13         14      반품 신청 과정이 복잡해서 이용하기 어려웠습니다  교환/환불/반품 문제  0.130
카

In [10]:
def get_top_keywords(row, feature_names, top_n=3):
    """TF-IDF 점수가 높은 상위 n개 키워드를 추출해 문자열로 반환"""
    word_score_list = []
    for word, score in zip(feature_names, row):
        if score > 0:                              # 해당 리뷰에 등장한 단어만 대상
            word_score_list.append((word, score))

    # TF-IDF 점수 높은 순으로 정렬
    word_score_list = sorted(word_score_list, key=lambda x: x[1], reverse=True)

    # 상위 n개 키워드를 문자열로 반환 (예: "배송이, 포장이, 만족")
    top_keywords = [word for word, score in word_score_list[:top_n]]
    return ', '.join(top_keywords)


keyword_results = []

for i in range(tfidf_df.shape[0]):
    keywords = get_top_keywords(
        tfidf_df.iloc[i],    # i번째 리뷰의 TF-IDF 점수 행
        tfidf_words,         # 전체 단어 목록
        top_n=3
    )
    keyword_results.append({
        'review_id': review_df.loc[i, 'review_id'],
        'review_text': review_df.loc[i, 'review_text'],
        '핵심 키워드': keywords    # ★ 띄어쓰기 포함 (merge 시 컬럼명 일치 필요)
    })

keyword_df = pd.DataFrame(keyword_results)
print('리뷰별 핵심 키워드')
print(keyword_df)

리뷰별 핵심 키워드
    review_id                     review_text              핵심 키워드
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다   깔끔해서, 빠르고, 만족했습니다
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다      가격이, 비쌌습니다, 조금
2           3           주문한 색상과 다른 제품이 도착했습니다        다른, 색상과, 주문한
3           4             고객센터 응답이 늦어서 불편했습니다    늦어서, 불편했습니다, 응답이
4           5          교환 처리가 빠르게 진행되어 만족했습니다       교환, 빠르게, 진행되어
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다      결제가, 실패해서, 앱에서
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다      들었습니다, 마음에, 만큼
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다     날짜를, 맞추지, 못했습니다
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다  걸려서, 너무, 불만족스러웠습니다
9          10        제품 설명과 실제 상품이 달라서 실망했습니다       달라서, 상품이, 설명과
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다      결제, 금액이, 나왔습니다
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다       상태로, 일부가, 파손된
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다    문제를, 상담원이, 주었습니다
13         14      반품 신청 과정이 복잡해서 이용하기 어려웠습니다       과정이, 반품, 복잡해서

In [11]:
# 모든 리뷰에서 단어별 TF-IDF 점수를 합산해 전체 중요도 계산
word_total_scores = tfidf_df.sum(axis=0)  # axis=0: 행(리뷰) 방향으로 합산

top_words_df = pd.DataFrame({
    '단어': word_total_scores.index,
    '전체 중요도': word_total_scores.values
})
top_words_df = top_words_df.sort_values(by='전체 중요도', ascending=False).head(10)
print(top_words_df.round(3))

        단어  전체 중요도
20  만족했습니다   0.828
4     고객센터   0.808
26     배송이   0.781
64     처리가   0.769
56     제품이   0.769
68     포장이   0.766
16  도착했습니다   0.754
35      상품   0.707
12     늦어서   0.516
49     응답이   0.516


In [12]:
# 분류 결과 + 핵심 키워드를 review_id 기준으로 병합
final_report_df = classification_df.merge(
    keyword_df[['review_id', '핵심 키워드']], on='review_id'
)
print(final_report_df)

    review_id                     review_text         분류결과    유사도  \
0           1         배송이 빠르고 포장이 깔끔해서 만족했습니다     배송/포장 문제  0.000   
1           2        상품 품질은 좋았지만 가격이 조금 비쌌습니다  상품 품질/정보 문제  0.104   
2           3           주문한 색상과 다른 제품이 도착했습니다     배송/포장 문제  0.000   
3           4             고객센터 응답이 늦어서 불편했습니다     배송/포장 문제  0.000   
4           5          교환 처리가 빠르게 진행되어 만족했습니다  교환/환불/반품 문제  0.150   
5           6      앱에서 결제가 자꾸 실패해서 주문을 못 했습니다     배송/포장 문제  0.000   
6           7       재구매하고 싶을 만큼 제품이 마음에 들었습니다     배송/포장 문제  0.000   
7           8       배송이 지연되어 선물 날짜를 맞추지 못했습니다     배송/포장 문제  0.000   
8           9      환불 처리가 너무 오래 걸려서 불만족스러웠습니다  교환/환불/반품 문제  0.271   
9          10        제품 설명과 실제 상품이 달라서 실망했습니다  상품 품질/정보 문제  0.127   
10         11   쿠폰 적용이 되지 않아 결제 금액이 다르게 나왔습니다     결제/주문 문제  0.219   
11         12  포장이 훼손되어 상품 일부가 파손된 상태로 도착했습니다  상품 품질/정보 문제  0.099   
12         13    고객센터 상담원이 친절하게 문제를 해결해 주었습니다     배송/포장 문제  0.000   
13         14      반품 신청 과정이 복잡해서 